In [1]:
from pathlib import Path

data_dir = Path(r"C:\Users\hp\Downloads\archive (4)")

csv_files = list(data_dir.rglob("*.csv"))

for file in csv_files:
    print(file)

C:\Users\hp\Downloads\archive (4)\er_data.csv
C:\Users\hp\Downloads\archive (4)\post_intervention_er_data.csv


In [2]:
from pathlib import Path
import pandas as pd

data_dir = Path(r"C:\Users\hp\Downloads\archive (4)")

baseline = pd.read_csv(data_dir / "er_data.csv")
post_intervention = pd.read_csv(
    data_dir / "post_intervention_er_data.csv"
)

baseline["dataset_phase"] = "Before intervention"
post_intervention["dataset_phase"] = "After intervention"

er_data = pd.concat(
    [baseline, post_intervention],
    ignore_index=True,
)

print("Baseline records:", len(baseline))
print("Post-intervention records:", len(post_intervention))
print("Combined records:", len(er_data))
print("\nColumns:", er_data.columns.tolist())

display(er_data.head())

print("\nMissing values:")
print(er_data.isna().sum())

print("\nWait-time summary by phase:")
display(
    er_data.groupby("dataset_phase")["WaitTime_Mins"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nTriage distribution by phase:")
display(
    pd.crosstab(
        er_data["Triage_Level"],
        er_data["dataset_phase"],
    )
)

Baseline records: 250
Post-intervention records: 250
Combined records: 500

Columns: ['CaseID', 'Shift', 'TriageLevel', 'StaffOnDuty', 'WaitTime_Mins', 'Walkout_YN', 'ArrivalTime', 'Age', 'CriticalCases_OnShift', 'dataset_phase', 'Defect_YN']


,CaseID,Shift,TriageLevel,StaffOnDuty,WaitTime_Mins,Walkout_YN,ArrivalTime,Age,CriticalCases_OnShift,dataset_phase,Defect_YN
0,1,Day,2,5,160.12,No,Afternoon,45.3,2,Before intervention,NaN
1,2,Night,1,6,190.45,Yes,Morning,50.1,1,Before intervention,NaN
2,3,Day,3,5,130.78,No,Evening,39.7,0,Before intervention,NaN
3,4,Day,2,5,175.23,No,Afternoon,48.9,3,Before intervention,NaN
4,5,Night,2,4,210.67,Yes,Evening,55.2,2,Before intervention,NaN



Missing values:
CaseID                     0
Shift                      0
TriageLevel                0
StaffOnDuty                0
WaitTime_Mins              0
Walkout_YN                 0
ArrivalTime                0
Age                        0
CriticalCases_OnShift      0
dataset_phase              0
Defect_YN                250
dtype: int64

Wait-time summary by phase:


,count,mean,median,min,max
dataset_phase,,,,,
After intervention,250,127.92,129.42,64.29,221.65
Before intervention,250,169.06,170.34,125.45,210.89



Triage distribution by phase:


KeyError: 'Triage_Level'

In [3]:
print("Triage distribution by phase:")
display(
    pd.crosstab(
        er_data["TriageLevel"],
        er_data["dataset_phase"],
    )
)

print("\nDefect values by phase:")
display(
    pd.crosstab(
        er_data["dataset_phase"],
        er_data["Defect_YN"].fillna("Not recorded"),
    )
)

print("\nWait time by triage level:")
display(
    er_data.groupby("TriageLevel")["WaitTime_Mins"]
    .agg(["count", "mean", "median"])
    .round(2)
)

print("\nWait time by staff on duty:")
display(
    er_data.groupby("StaffOnDuty")["WaitTime_Mins"]
    .agg(["count", "mean", "median"])
    .round(2)
)

print("\nWait time by critical cases on shift:")
display(
    er_data.groupby("CriticalCases_OnShift")["WaitTime_Mins"]
    .agg(["count", "mean", "median"])
    .round(2)
)

Triage distribution by phase:


dataset_phase,After intervention,Before intervention
TriageLevel,,
1,63,63
2,125,125
3,62,62



Defect values by phase:


Defect_YN,No,Not recorded,Yes
dataset_phase,,,
After intervention,213,0,37
Before intervention,0,250,0



Wait time by triage level:


,count,mean,median
TriageLevel,,,
1,126,161.19,175.34
2,250,149.19,160.12
3,124,134.20,135.45



Wait time by staff on duty:


,count,mean,median
StaffOnDuty,,,
4,100,145.17,135.12
5,324,146.98,146.18
6,76,159.30,177.62



Wait time by critical cases on shift:


,count,mean,median
CriticalCases_OnShift,,,
0,124,134.20,135.45
1,124,142.59,160.12
2,128,155.56,173.01
3,124,161.40,177.68


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

pre_intervention = er_data[
    er_data["dataset_phase"].eq("Before intervention")
].copy()

feature_columns = [
    "Shift",
    "TriageLevel",
    "StaffOnDuty",
    "ArrivalTime",
    "CriticalCases_OnShift",
]

X = pre_intervention[feature_columns]
y = pre_intervention["WaitTime_Mins"]

categorical_features = [
    "Shift",
    "TriageLevel",
    "ArrivalTime",
]

numeric_features = [
    "StaffOnDuty",
    "CriticalCases_OnShift",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features,
        ),
        ("numeric", "passthrough", numeric_features),
    ]
)

models = {
    "Median baseline": DummyRegressor(strategy="median"),
    "Random Forest": Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    min_samples_leaf=4,
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2",
        },
    )

    results.append(
        {
            "model": model_name,
            "CV MAE": -scores["test_mae"].mean(),
            "CV RMSE": -scores["test_rmse"].mean(),
            "CV R2": scores["test_r2"].mean(),
        }
    )

display(
    pd.DataFrame(results).round(2)
)

,model,CV MAE,CV RMSE,CV R2
0,Median baseline,20.89,25.21,-0.10
1,Random Forest,3.59,5.21,0.95


In [7]:
post_data = er_data[
    er_data["dataset_phase"].eq("After intervention")
].copy()

X_train = pre_intervention[feature_columns]
y_train = pre_intervention["WaitTime_Mins"]

X_post = post_data[feature_columns]
y_post = post_data["WaitTime_Mins"]

random_forest_model = models["Random Forest"]

random_forest_model.fit(X_train, y_train)

post_predictions = random_forest_model.predict(X_post)

print(
    "Before → After test MAE:",
    round(mean_absolute_error(y_post, post_predictions), 2),
    "minutes",
)

print(
    "Before → After test RMSE:",
    round(mean_squared_error(y_post, post_predictions) ** 0.5, 2),
    "minutes",
)

print(
    "Before → After test R²:",
    round(r2_score(y_post, post_predictions), 4),
)

comparison = post_data[
    [
        "Shift",
        "TriageLevel",
        "StaffOnDuty",
        "ArrivalTime",
        "CriticalCases_OnShift",
        "WaitTime_Mins",
    ]
].copy()

comparison["Predicted_WaitTime_Mins"] = post_predictions
comparison["Absolute_Error_Mins"] = (
    comparison["WaitTime_Mins"]
    - comparison["Predicted_WaitTime_Mins"]
).abs()

display(comparison.head(10).round(2))

Before → After test MAE: 45.92 minutes
Before → After test RMSE: 53.98 minutes
Before → After test R²: -4.3045


,Shift,TriageLevel,StaffOnDuty,ArrivalTime,CriticalCases_OnShift,WaitTime_Mins,Predicted_WaitTime_Mins,Absolute_Error_Mins
250,Day,2,5,Afternoon,2,140.06,176.44,36.39
251,Night,1,6,Morning,1,124.62,166.49,41.87
252,Day,3,5,Evening,0,143.73,136.77,6.96
253,Day,2,5,Afternoon,3,165.01,178.60,13.59
254,Night,2,4,Evening,2,122.29,192.70,70.41
255,Day,1,5,Morning,1,122.29,160.44,38.15
256,Day,2,5,Afternoon,2,166.37,176.44,10.07
257,Night,3,4,Evening,0,146.64,133.72,12.92
258,Day,2,5,Morning,1,116.57,161.79,45.22
259,Night,1,6,Afternoon,3,141.17,199.97,58.80


## Conclusion: Simulated ER Wait-Time Dataset

- The dataset contains 500 simulated emergency-room records: 250 before and 250 after an operational intervention.
- Triage level, staffing, shift, arrival period, and critical-case load were evaluated as operational predictors.
- `CaseID`, `Walkout_YN`, `dataset_phase`, and `Defect_YN` were excluded from prediction because they are identifiers, outcomes, or intervention/leakage indicators.
- Random Forest achieved an apparently strong five-fold cross-validation result on pre-intervention data: MAE 3.59 minutes and R² 0.95.
- However, when trained on pre-intervention records and tested on post-intervention records, performance collapsed to MAE 45.92 minutes, RMSE 53.98 minutes, and R² -4.3045.
- This shows that the model learned the synthetic pre-intervention data-generation pattern and did not generalize after the operational process changed.
- The dataset is useful for demonstrating dataset shift and model-generalization failure, but it is rejected for SmartCare wait-time prediction.
- No model from this dataset is saved, integrated, or deployed.